In [0]:
sleep_bronze = spark.table(
    "health_and_habit_guardian.bronze.sleep"
)

In [0]:
from pyspark.sql import functions as F

# explicitly cast types in the Silver layer to estabilish a canonical schema

sleep = (
    sleep_bronze
    .withColumn(
        "sleep_duration_hours",
        F.col("sleep_duration_hours").cast("float")
    )
    .withColumn(
        "deep_sleep_hours",
        F.col("deep_sleep_hours").cast("float")
    )
    .withColumn(
        "light_sleep_hours",
        F.col("light_sleep_hours").cast("float")
    )
    .withColumn(
        "rem_sleep_hours",
        F.col("rem_sleep_hours").cast("float")
    )
    .withColumn(
        "awake_hours",
        F.col("awake_hours").cast("float")
    )
    .withColumn(
        "awakenings",
        F.col("awakenings").cast("int")
    )
)

In [0]:
# determine what constitutes as a valid sleep record

sleep = (
    sleep
    .withColumn(
        "missing_required",
        (
            F.col("user_id").isNull() |
            F.col("date").isNull() |
            F.col("sleep_start").isNull() |
            F.col("sleep_end").isNull()
        )
    )
    .withColumn(
        "invalid_sleep_time",
        F.col("sleep_end") <= F.col("sleep_start")
    )
    .withColumn(
        "invalid_sleep_duration",
        F.col("sleep_duration_hours").isNull() |
        (F.col("sleep_duration_hours") < 0)
    )
    .withColumn(
        "invalid_deep_sleep",
        F.col("deep_sleep_hours").isNull() |
        (F.col("deep_sleep_hours") < 0)
    )
    .withColumn(
        "invalid_light_sleep",
        F.col("light_sleep_hours").isNull() |
        (F.col("light_sleep_hours") < 0)
    )
    .withColumn(
        "invalid_rem_sleep",
        F.col("rem_sleep_hours").isNull() |
        (F.col("rem_sleep_hours") < 0)
    )
    .withColumn(
        "invalid_awake_hours",
        F.col("awake_hours").isNull() |
        (F.col("awake_hours") < 0)
    )
    .withColumn(
        "invalid_awakenings",
        F.col("awakenings").isNull() |
        (F.col("awakenings") < 0)
    )
)

In [0]:
# validate that sleep stage components approximately add up to the reported sleep duration

sleep = (
    sleep
    .withColumn(
        "sleep_stage_total",
        F.coalesce(F.col("deep_sleep_hours"), F.lit(0.0)) +
        F.coalesce(F.col("light_sleep_hours"), F.lit(0.0)) +
        F.coalesce(F.col("rem_sleep_hours"), F.lit(0.0))
    )
    .withColumn(
        "_stage_duration_mismatch",
        F.abs(
            F.col("sleep_stage_total") -
            F.col("sleep_duration_hours")
        ) > 0.25
    )
)

In [0]:
# calculate time in bed

sleep = sleep.withColumn(
    "time_in_bed_hours",
    (
        F.col("sleep_end").cast("long") - F.col("sleep_start").cast("long")
    ) / 3600.0
)

In [0]:
# calculate sleep efficiency

sleep = sleep.withColumn(
    "sleep_efficiency",
    F.when(
        F.col("time_in_bed_hours") > 0,
        F.col("sleep_duration_hours") /
        F.col("time_in_bed_hours")
    )
)

sleep = sleep.withColumn(
    "_invalid_efficiency",
    (F.col("sleep_efficiency") <= 0) |
    (F.col("sleep_efficiency") > 1.01)
)

In [0]:
# if there are multiple sleep records for the same user/day, keep the latest ingestion

from pyspark.sql.window import Window

window = Window.partitionBy(
    "user_id",
    "date"
).orderBy(
    F.col("_ingested_at").desc()
)

sleep_deduped = (
    sleep
    .withColumn(
        "_row_number",
        F.row_number().over(window)
    )
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)


In [0]:
# add a quality status

sleep = sleep.withColumn(
    "_quality_status",
    F.when(
        F.col("missing_required") |
        F.col("invalid_sleep_time") |
        F.col("invalid_sleep_duration") |
        F.col("invalid_deep_sleep") |
        F.col("invalid_light_sleep") |
        F.col("invalid_rem_sleep") |
        F.col("invalid_awake_hours") |
        F.col("invalid_awakenings") |
        F.col("_stage_duration_mismatch"),
        F.lit("INVALID")
    ).otherwise(F.lit("VALID"))
)


In [0]:
sleep = sleep.withColumn(
    "_processed_at",
    F.current_timestamp()
)

In [0]:
valid_sleep = (
    sleep
    .filter(F.col("_quality_status") == "VALID")
)

invalid_sleep = (
    sleep
    .filter(F.col("_quality_status") == "INVALID")
)

In [0]:
(
    invalid_sleep
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(
        "health_and_habit_guardian.silver.quarantine_sleep"
    )
)

In [0]:
(
    valid_sleep
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "health_and_habit_guardian.silver.sleep"
    )
)